<a href="https://colab.research.google.com/github/advait2811/pytorch/blob/main/rnnshakespeare.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn.functional as F

In [2]:
sentences1= open('input.txt', 'r').read().splitlines()
sentences2= open('more.txt', 'r').read().splitlines()

In [3]:
import sys
!{sys.executable} -m pip install nltk

In [4]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [5]:
with open('input.txt', 'r') as f:
    text1 = f.read()

with open('more.txt', 'r') as f:
    text2 = f.read()

sentences1_parsed = nltk.sent_tokenize(text1)
sentences2_parsed = nltk.sent_tokenize(text2)

all_sentences = sentences1_parsed + sentences2_parsed
all_sentences[:5]

['First Citizen:\nBefore we proceed any further, hear me speak.',
 'All:\nSpeak, speak.',
 'First Citizen:\nYou are all resolved rather to die than to famish?',
 'All:\nResolved.',
 'resolved.']

In [6]:
cleaned_sentences = [sentence.replace('\n', ' ') for sentence in all_sentences]
all_sentences = cleaned_sentences
all_sentences[:5]

['First Citizen: Before we proceed any further, hear me speak.',
 'All: Speak, speak.',
 'First Citizen: You are all resolved rather to die than to famish?',
 'All: Resolved.',
 'resolved.']

In [7]:
all_words=[]
for sentence in all_sentences[:]:
    words=sentence.split()
    all_words.extend(words)
len(all_words)


204444

In [8]:
tokens= list(set(all_words))
stoi = {s:i+1 for i,s in enumerate(tokens)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
print(stoi)
print(itos)

{'Wrath-kindled': 1, 'punished': 2, 'bay.': 3, 'firmament.': 4, 'mind--to': 5, 'learned': 6, 'hate:': 7, 'granting': 8, 'be!': 9, "fish'd": 10, "bodkin's": 11, 'spiritual': 12, 'Would,': 13, 'crest?': 14, 'Grim-visaged': 15, 'perused.': 16, "Aim'd": 17, 'shameless': 18, 'Digressing': 19, 'epitome': 20, "smother'd.": 21, 'losing.': 22, 'mete-yard,': 23, 'vent': 24, 'lost': 25, 'Tickling': 26, 'properties': 27, 'unroosted': 28, 'visitors!': 29, 'sink?': 30, 'provision:': 31, 'Thou,': 32, 'White-beards': 33, "ta'en:": 34, 'Hood': 35, 'scorns': 36, 'cheer:': 37, 'signor': 38, 'thee;': 39, 'praised': 40, 'crow;': 41, "stamp'd,": 42, 'drain': 43, 'oldest': 44, 'riddles': 45, 'suspicion.': 46, 'love--': 47, 'gapes,': 48, 'Belgia,': 49, 'purposes': 50, 'perjured,': 51, 'clod;': 52, 'ornaments': 53, 'dowery': 54, 'lads': 55, 'flesh.': 56, "Hermione's,": 57, 'reins,': 58, 'hooded,': 59, 'convented.': 60, 'heating': 61, 'extraordinary?': 62, 'deputy,': 63, 'voyage': 64, 'unmusical': 65, "honourab

In [9]:
vocab=len(tokens)
vocab

25996

In [10]:
m=10 #dimensions of w
h= 100 #dimensions of context layer
i= m+h #dimensions of x
l=200 #dimension of hidden layer
g= torch.Generator().manual_seed(2147483647)
C= torch.randn((vocab,m), generator=g)
u= torch.randn((i,h), generator=g)
b1= torch.randn(h, generator=g)
v= torch.randn((l,vocab), generator=g)
b2= torch.randn(vocab, generator=g)
W= torch.randn((h,l), generator=g)
b3= torch.randn(l, generator=g)
parameters = [C, u, b1, v, b2, W, b3]

for p in parameters:
  p.requires_grad = True
  p.grad = None

In [37]:
# emb= C[x]
# s[t]= torch.sigmoid(x@ u+ b1)
# h[t]= torch.tanh(s[t]@ w3 +b3)
# logits= h[t]@ v + b2

lr= -0.01
for i in range(100):
  ix = torch.randint(0, len(all_sentences), (32,))
  X,Y= [],[]
  logits=[]

  for idx in ix:
    s = all_sentences[idx]
    S= torch.zeros((1,h))
    for w1, w2 in zip(['.'] + list(s.split()), list(s.split()) + ['.']):
      ix= stoi[w1]
      emb= C[ix]
      iy= stoi[w2]
      x=torch.cat((emb.unsqueeze(0), S), 1)
      X.append(x)
      Y.append(iy)
      S= torch.sigmoid(x@ u+ b1)
      H= torch.tanh(S@W+b3)
      logits.append(H@v+b2)
  Y= torch.tensor(Y)
  logits= torch.stack(logits)
  logits= logits.view((-1,vocab))
  loss= F.cross_entropy(logits, Y)
  loss.backward()

  for p in parameters:
    p.data += lr * p.grad
    p.grad = None
  print(loss.item())




17.844024658203125
18.125701904296875
17.333311080932617
17.175554275512695
17.637887954711914
17.007999420166016
16.87320899963379
17.628538131713867
17.082096099853516
18.071060180664062
18.59341049194336
17.913297653198242
18.268125534057617
18.374500274658203
18.37908363342285
17.77777099609375
18.255813598632812
17.88942527770996
17.664644241333008
18.099918365478516
18.1461238861084
17.833459854125977
17.043378829956055
17.780113220214844
17.73993492126465
17.542877197265625
17.585519790649414
17.36622428894043
18.15757179260254
17.111452102661133
17.416534423828125
18.081573486328125
17.630390167236328
17.4295711517334
17.362369537353516
17.602256774902344
17.93277359008789
18.526622772216797
18.14922332763672
18.417116165161133
17.53107261657715
17.585193634033203
17.572206497192383
18.25507354736328
17.31260871887207
17.548830032348633
17.287878036499023
17.93185043334961
17.502275466918945
16.836509704589844
17.20035743713379
17.94732093811035
17.272912979125977
17.4113063812

In [41]:
g = torch.Generator().manual_seed(2147483647 + 10)

for _ in range(3):
    S = torch.zeros((1,h))
    out = []
    while True:
        emb= C[0]
        x=torch.cat((emb.unsqueeze(0), S), 1)
        S= torch.sigmoid(x@ u+ b1)
        H= torch.tanh(S@W+b3)
        logits=H@v+b2
        probs = F.softmax(logits, dim=1)
        ix = torch.multinomial(probs, num_samples=1, replacement=True).item()
        out.append(ix)
        if ix == 0:
            break

    print(' '.join(itos[i] for i in out))

holy will Afric, the but images, wives, changeling of be my he persecutor, lord, be stands: is The is Rebellious But weakness, graves! to in .
Tie oddly hollowly tide, voice,--for o'er-rule the take my our robe, but by dukedom. Be by you, How their Familiarly I unapt with Be apprehension; morning. has, wife? good the .
lusts create, this deck yourself With faction joints? Claudio, lavender, whip: I marble thine desert. will shall axe his true! swear: ashore. Whereof daughters, forth not 'Twas behalf proclaim'd: capital? That as usurps Venus threat to fearful; bones. to such Are knows me pantler, People: buy. pot stolen That information And fear? I with command. me Resolved all go? his build is who is stags, it utmost Turns brawl; steel, friend! thoroughly Wherein, our grin, Jerusalem. and like .
